## __str__ vs __repr__ — 4 cases

- `print(x)` / `str(x)` always calls `x.__str__()`. If not overridden, falls back to `object.__str__`, which internally does `return self.__repr__()`.
- bare `x` in REPL / `repr(x)` always calls `x.__repr__()`. If not overridden, falls back to `object.__repr__` (ugly default), with no further fallback to `__str__`.

### Case 1: Neither overridden

In [ ]:
class Neither:
    pass

n = Neither()
print(n)   # <__main__.Neither object at 0x...>

In [ ]:
n   # bare -> repr -> also ugly default

### Case 2: Only __repr__ overridden

In [ ]:
class OnlyRepr:
    def __repr__(self):
        return "OnlyRepr's repr"

o = OnlyRepr()
print(o)   # "OnlyRepr's repr" -- via object.__str__ delegating to __repr__

In [ ]:
o   # bare -> repr -> "OnlyRepr's repr"

### Case 3: Only __str__ overridden

In [ ]:
class OnlyStr:
    def __str__(self):
        return "OnlyStr's str"

s = OnlyStr()
print(s)   # "OnlyStr's str"

In [ ]:
s   # bare -> repr -> no __repr__ defined -> ugly default (str is NOT used as fallback here)

### Case 4: Both overridden

In [ ]:
class Both:
    def __repr__(self):
        return "Both's repr"
    def __str__(self):
        return "Both's str"

b = Both()
print(b)   # "Both's str"

In [ ]:
b   # bare -> repr -> "Both's repr"

## __eq__

In [ ]:
class Point:
    def __init__(self, x, y):
        self.x, self.y = x, y

p1 = Point(1, 2)
p2 = Point(1, 2)
p1 == p2      # False — different objects in memory, even though x/y match
p1 == p1      # True — same object


#### p1 == p2 actually calls p1.__eq__(p2). Since Point doesn't define __eq__, it uses object.__eq__, whose body is essentially return self is other — pure identity check.

In [8]:
class Point:
    def __init__(self, x, y):
        self.x, self.y = x, y

    def __eq__(self, other):
        return self.x == other.x and self.y == other.y


In [ ]:
p1 = Point(1, 2)
p2 = Point(1, 2)
p1 == p2      # True now — your __eq__ runs instead of object's default


### __hash__

object's default hash is based on identity (id(obj)), matching its default identity-based __eq__. The moment you override __eq__ to be value-based, that guarantee is broken — two "equal" objects (by your new rule) would still have different default hashes. So Python protects you: defining __eq__ automatically sets __hash__ to None on your class, making instances unhashable.

In [ ]:
class Point:
    def __init__(self, x, y):
        self.x, self.y = x, y

    def __eq__(self, other):
        return self.x == other.x and self.y == other.y

p1 = Point(1, 2)
hash(p1)          # TypeError: unhashable type: 'Point'
{p1, p1}           # TypeError — can't put it in a set either


##### Fix: define __hash__ yourself, consistent with __eq__ — same fields go into both:

In [11]:
class Point:
    def __init__(self, x, y):
        self.x, self.y = x, y

    def __eq__(self, other):
        return self.x == other.x and self.y == other.y

    def __hash__(self):
        return hash((self.x, self.y))   # combine fields into a tuple, hash that


In [12]:
p1 = Point(1, 2)
p2 = Point(1, 2)
hash(p1) == hash(p2)     # True — required, since p1 == p2
{p1, p2}                  # {Point instance} -- set sees them as duplicates, keeps one


{<__main__.Point at 0x10b44f770>}

#### Rule of thumb: whatever fields you compare in __eq__, hash those same fields in __hash__

### len

In [13]:
class Cart:
    def __init__(self):
        self.items = []

    def __len__(self):
        return len(self.items)

c = Cart()
c.items = ["apple", "banana", "cherry"]
len(c)      # 3 — calls c.__len__()


3

If you don't define __len__, len(obj) raises TypeError: object of type 'Cart' has no len(). One extra rule: __len__ must return a non-negative int — if you return something else (like a float or negative number), Python raises an error. Also, bool(obj) uses __len__ as a fallback when __bool__ isn't defined — an object with len(obj) == 0 is treated as falsy (if not cart:), same as empty lists/dicts.

## __getitem__ / __setitem__ / __contains__

- `obj[i]` calls `obj.__getitem__(i)`
- `obj[i] = value` calls `obj.__setitem__(i, value)`
- `x in obj` calls `obj.__contains__(x)`

Fallback rule: if `__contains__` is missing but `__getitem__` exists, `in` falls back to manually iterating `obj[0], obj[1], ...` until a match or an `IndexError`.

In [ ]:
class Cart:
    def __init__(self):
        self.items = []

    def __getitem__(self, index):
        return self.items[index]

    def __setitem__(self, index, value):
        self.items[index] = value

    def __contains__(self, item):
        return item in self.items

c = Cart()
c.items = ["apple", "banana"]
c[0]                  # "apple"

In [ ]:
c[0] = "mango"   # calls c.__setitem__(0, "mango")
c.items

In [ ]:
"mango" in c        # True -> calls c.__contains__("mango")
"cherry" in c        # False